In [1]:
# Pipeline Configuration
RUN_MODE = "production"  # "test" or "production"
TEST_SAMPLE_SIZE = 10
MAX_WORKERS = 4
CHUNK_SIZE = 500
MAX_RETRIES = 5
TIMEOUT = 30
ENABLE_CACHE = False
ENABLE_CHECKPOINT = True
SAVE_PREVIEW_IMAGES = False
PIPELINE_VERSION = "1.0.0"


# !pip -q install geopandas rasterio rioxarray pystac-client planetary-computer odc-stac shapely pyproj xarray folium leafmap



In [2]:
import geopandas as gpd
import pandas as pd
import numpy as np

from shapely.geometry import Point

import rasterio
import rioxarray

import planetary_computer
import pystac_client



In [3]:
import os

folders = [
    "../data",
    "../data/raw",
    "../data/processed",
    "../data/features",
    "../data/metadata",
    "../data/final",
    "../data/lucas",
    "../data/sentinel",
    "../data/weather",
    "../data/soilgrids",
    "../outputs",
    "../outputs/csv",
    "../outputs/maps",
    "../outputs/figures",
    "../outputs/reports",
    "../outputs/metrics",
    "../outputs/learning_curves",
    "../outputs/feature_importance",
    "../outputs/confusion_matrix",
    "../models",
    "../models/machine_learning",
    "../models/deep_learning",
    "../models/ensemble",
    "../models/best",
    "../models/experimental"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"Created: {folder}")




Created: ../data
Created: ../data/raw
Created: ../data/processed
Created: ../data/features
Created: ../data/metadata
Created: ../data/final
Created: ../data/lucas
Created: ../data/sentinel
Created: ../data/weather
Created: ../data/soilgrids
Created: ../outputs
Created: ../outputs/csv
Created: ../outputs/maps
Created: ../outputs/figures
Created: ../outputs/reports
Created: ../outputs/metrics
Created: ../outputs/learning_curves
Created: ../outputs/feature_importance
Created: ../outputs/confusion_matrix
Created: ../models
Created: ../models/machine_learning
Created: ../models/deep_learning
Created: ../models/ensemble
Created: ../models/best
Created: ../models/experimental


In [4]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

print("Connected Successfully")



Connected Successfully


# # collections = catalog.get_collections()

# # for c in collections:
    # # print(c.id)


print("Skipping collections listing for speed.")




from shapely.geometry import Point

lon = 31.083
lat = 30.563

point = Point(lon, lat)

buffer = point.buffer(0.001)

geometry = buffer.__geo_interface__

geometry



In [5]:
import numpy as np
import pandas as pd
import datetime
import os
import time
from shapely.geometry import Point
import rioxarray
from odc.stac import stac_load

# Configure GDAL HTTP timeouts to prevent indefinite hangs
os.environ["GDAL_HTTP_TIMEOUT"] = "15"
os.environ["GDAL_HTTP_MAX_RETRY"] = "3"
os.environ["GDAL_HTTP_RETRY_DELAY"] = "2"
os.environ["CPL_VSIL_CURL_ALLOWED_EXTENSIONS"] = ".tif,.tiff"

def extract_s2_features(dataset):
    blue    = dataset.B02.squeeze()
    green   = dataset.B03.squeeze()
    red     = dataset.B04.squeeze()
    rededge1 = dataset.B05.squeeze() # 705nm (B5)
    rededge2 = dataset.B06.squeeze() # 740nm (B6)
    rededge3 = dataset.B07.squeeze() # 783nm (B7)
    nir     = dataset.B08.squeeze() # 842nm (B8)
    nireg   = dataset.B8A.squeeze() # 865nm (B8A)
    swir1   = dataset.B11.squeeze() # 1610nm (B11)
    swir2   = dataset.B12.squeeze() # 2190nm (B12)

    features = {}

    features["NDVI"] = float(((nir - red) / (nir + red + 1e-8)).mean())
    features["EVI"] = float((2.5 * (nir - red) / (nir + 6 * red - 7.5 * blue + 1 + 1e-8)).mean())
    features["SAVI"] = float((1.5 * (nir - red) / (nir + red + 0.5 + 1e-8)).mean())
    features["MSAVI"] = float((((2 * nir + 1) - np.sqrt(np.maximum(0, (2 * nir + 1)**2 - 8 * (nir - red)))) / 2).mean())
    features["GNDVI"] = float(((nir - green) / (nir + green + 1e-8)).mean())
    features["NDMI"] = float(((nir - swir1) / (nir + swir1 + 1e-8)).mean())
    features["NDWI"] = float(((green - nir) / (green + nir + 1e-8)).mean())
    features["BSI"] = float((((swir1 + red) - (nir + blue)) / ((swir1 + red) + (nir + blue) + 1e-8)).mean())
    features["NDRE"] = float(((nir - rededge1) / (nir + rededge1 + 1e-8)).mean())
    features["CIre"] = float(((nir / (rededge1 + 1e-8)) - 1).mean())
    features["Brightness"] = float(np.sqrt((red**2 + nir**2) / 2).mean())
    features["ClayIndex"] = float((swir1 / (swir2 + 1e-8)).mean())

    # New indices
    features["IRECI"] = float((((rededge3 - red) * rededge2) / (rededge1 + 1e-8)).mean())
    features["S2REP"] = float((705 + 35 * (0.5 * (rededge3 + red) - rededge1) / (rededge2 - rededge1 + 1e-8)).mean())
    features["MTCI"] = float(((rededge2 - rededge1) / (rededge1 - red + 1e-8)).mean())
    features["NIRv"] = float((((nir - red) / (nir + red + 1e-8)) * nir).mean())
    features["NBR"] = float(((nir - swir2) / (nir + swir2 + 1e-8)).mean())
    features["RedEdge_NDVI"] = float(((nir - rededge1) / (nir + rededge1 + 1e-8)).mean())

    return pd.DataFrame([features])

def process_point(row):
    point_id = int(row["POINT_ID"])
    lat = float(row["Latitude"])
    lon = float(row["Longitude"])
    date_str = str(row["Survey_Date"])

    start = time.time()
    print("=" * 60)
    print(f"Starting POINT_ID={point_id}")

    meta = {
        "POINT_ID": point_id, "Latitude": lat, "Longitude": lon, "Survey_Date": date_str,
        "Extraction_Time": datetime.datetime.now().isoformat(), "Pipeline_Version": PIPELINE_VERSION,
        "Sentinel_Product": "sentinel-2-l2a", "Source_Dataset": "Sentinel-2 L2A",
        "sentinel_ok": np.nan
    }

    patch_tif = f"../data/features/image_patches/s2/s2_{point_id}.tif"
    bands = ["B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B11", "B12"]

    try:
        cache_file_valid = False
        if ENABLE_CACHE and os.path.exists(patch_tif):
            try:
                import rasterio
                with rasterio.open(patch_tif) as src:
                    num_bands = src.count
                if num_bands == 10:
                    cache_file_valid = True
                else:
                    try:
                        os.remove(patch_tif)
                    except Exception:
                        pass
            except Exception:
                try:
                    if os.path.exists(patch_tif):
                        os.remove(patch_tif)
                except Exception:
                    pass

        if cache_file_valid:
            rds = rioxarray.open_rasterio(patch_tif)
            meta_crs = str(rds.rio.crs)
            meta["Sentinel_Tile_ID"] = "cached"
            meta["S2_PRODUCT_ID"] = "cached"
            dataset = rds.to_dataset(dim="band").rename({i+1: b for i, b in enumerate(bands)})
        else:
            point = Point(lon, lat)
            buffer = point.buffer(0.001)
            geometry = buffer.__geo_interface__

            dt = pd.to_datetime(date_str, dayfirst=True, errors="coerce")
            if pd.isna(dt):
                dt = pd.to_datetime("2018-07-15")

            # Search window: ±60 days
            start_date = dt - datetime.timedelta(days=60)
            end_date = dt + datetime.timedelta(days=60)

            print("Searching STAC...")
            search = catalog.search(
                collections=["sentinel-2-l2a"],
                intersects=geometry,
                datetime=f"{start_date.strftime('%Y-%m-%d')}/{end_date.strftime('%Y-%m-%d')}",
                query={"eo:cloud_cover": {"lt": 40}}
            )
            items = list(search.items())

            # If no scene, expand to ±120 days
            if not items:
                start_date = dt - datetime.timedelta(days=120)
                end_date = dt + datetime.timedelta(days=120)
                search = catalog.search(
                    collections=["sentinel-2-l2a"],
                    intersects=geometry,
                    datetime=f"{start_date.strftime('%Y-%m-%d')}/{end_date.strftime('%Y-%m-%d')}",
                    query={"eo:cloud_cover": {"lt": 40}}
                )
                items = list(search.items())

            if not items:
                raise ValueError(f"No S2 scenes found for {point_id}")
            print("STAC search finished")
            print(f"Items found = {len(items)}")

            # Sort key: nearest date, lowest cloud cover, newest timestamp
            def sort_key(item):
                item_dt = pd.to_datetime(item.datetime).tz_localize(None)
                days_diff = abs((item_dt - dt).days)
                cloud = item.properties.get("eo:cloud_cover", 100.0)
                return (days_diff, cloud, -item_dt.timestamp())

            best_item = sorted(items, key=sort_key)[0]
            meta["Sentinel_Tile_ID"] = best_item.properties.get("s2:mgrs_tile", "")
            meta["S2_PRODUCT_ID"] = best_item.id

            # Retry loop for STAC loading
            dataset = None
            last_err = None
            print("Loading raster...")
            print("Calling stac_load...")
            t0 = time.time()
            for retry in range(MAX_RETRIES + 1):
                try:
                    dataset = stac_load(
                        [best_item], bands=bands, geopolygon=geometry, resolution=10, chunks={}
                    )
                    break
                except Exception as e:
                    last_err = e
                    time.sleep(2 ** retry)
            if dataset is None:
                raise last_err
            print(f"stac_load finished in {time.time()-t0:.2f} sec")
            print("Raster loaded")
            print(dataset)

            dataset = dataset.astype("float32") / 10000.0
            ds_squeezed = dataset.isel(time=0)
            meta_crs = str(dataset.rio.crs)

            # Save patch cache GeoTIFF
            stacked_da = ds_squeezed.to_array()
            stacked_da.rio.to_raster(patch_tif)
            dataset = ds_squeezed

        print("Computing NDVI...")
        df_feat = extract_s2_features(dataset)
        features = df_feat.iloc[0].to_dict()
        meta.update(features)
        meta["sentinel_ok"] = 1.0
        meta["CRS"] = meta_crs
        print("Features extracted")
        
        elapsed = time.time() - start
        print(f"Finished POINT_ID={point_id}")
        print(f"Elapsed = {elapsed:.2f} sec")
        print("=" * 60)
        return {"status": "success", "data": meta}
    except Exception as e:
        err_msg = str(e)
        features = {
            "NDVI": np.nan, "EVI": np.nan, "SAVI": np.nan, "MSAVI": np.nan,
            "GNDVI": np.nan, "NDMI": np.nan, "NDWI": np.nan, "BSI": np.nan,
            "NDRE": np.nan, "CIre": np.nan, "Brightness": np.nan, "ClayIndex": np.nan,
            "IRECI": np.nan, "S2REP": np.nan, "MTCI": np.nan, "NIRv": np.nan,
            "NBR": np.nan, "RedEdge_NDVI": np.nan
        }
        meta.update(features)
        meta["sentinel_ok"] = 0.0
        meta["CRS"] = np.nan
        meta["Sentinel_Tile_ID"] = np.nan
        meta["S2_PRODUCT_ID"] = np.nan
        
        elapsed = time.time() - start
        print(f"Finished POINT_ID={point_id} (FAILED) -> {err_msg}")
        print(f"Elapsed = {elapsed:.2f} sec")
        print("=" * 60)
        return {"status": "success", "data": meta}


for i, item in enumerate(items):
    print("="*60)
    print("Item:", i)
    print("Date:", item.datetime)
    print("Cloud Cover:", item.properties["eo:cloud_cover"])
    print("ID:", item.id)



best_item = sorted(
    items,
    key=lambda x: x.properties["eo:cloud_cover"]
)[0]

print(best_item.datetime)
print(best_item.properties["eo:cloud_cover"])



from odc.stac import stac_load



dataset = stac_load(
    [best_item],
    bands=["B02","B03","B04","B08","B11","B12"],
    geopolygon=geometry,
    resolution=10,
    chunks={}
)



dataset



# تحويل كل الباندات إلى Float Reflectance

dataset = dataset.astype("float32") / 10000

dataset



nir = dataset.B08
red = dataset.B04

ndvi = (nir - red) / (nir + red)

ndvi



print("Minimum :", float(ndvi.min()))
print("Maximum :", float(ndvi.max()))
print("Mean    :", float(ndvi.mean()))



import matplotlib.pyplot as plt

plt.figure(figsize=(8,8))

ndvi.squeeze().plot(
    cmap="RdYlGn",
    vmin=-1,
    vmax=1
)

plt.title("NDVI")
plt.axis("off")
plt.show()



blue = dataset.B02.squeeze()
green = dataset.B03.squeeze()
red = dataset.B04.squeeze()
nir = dataset.B08.squeeze()
swir1 = dataset.B11.squeeze()
swir2 = dataset.B12.squeeze()



import numpy as np

# Vegetation
NDVI = (nir - red) / (nir + red)

EVI = 2.5 * (
    (nir - red) /
    (nir + 6 * red - 7.5 * blue + 1)
)

SAVI = 1.5 * (
    (nir - red) /
    (nir + red + 0.5)
)

MSAVI = (
    (2 * nir + 1)
    - np.sqrt((2 * nir + 1) ** 2 - 8 * (nir - red))
) / 2

GNDVI = (nir - green) / (nir + green)

NDMI = (nir - swir1) / (nir + swir1)

NDWI = (green - nir) / (green + nir)

BSI = (
    (swir1 + red) - (nir + blue)
) / (
    (swir1 + red) + (nir + blue)
)



dataset = stac_load(
    [best_item],
    bands=["B02","B03","B04","B05","B08","B11","B12"],
    geopolygon=geometry,
    resolution=10,
    chunks={}
)

dataset = dataset.astype("float32") / 10000



rededge = dataset.B05.squeeze()

NDRE = (nir - rededge) / (nir + rededge)

CIre = (nir / rededge) - 1



Brightness = np.sqrt((red**2 + nir**2) / 2)

SoilColor = red / green

ClayIndex = swir1 / swir2



indices = {
    "NDVI": NDVI,
    "EVI": EVI,
    "SAVI": SAVI,
    "MSAVI": MSAVI,
    "GNDVI": GNDVI,
    "NDMI": NDMI,
    "NDWI": NDWI,
    "BSI": BSI,
    "NDRE": NDRE,
    "CIre": CIre,
    "Brightness": Brightness,
    "SoilColor": SoilColor,
    "ClayIndex": ClayIndex
}

for name, img in indices.items():
    print(
        name,
        "Min:", float(img.min()),
        "Max:", float(img.max()),
        "Mean:", float(img.mean())
    )



import matplotlib.pyplot as plt

for name, img in indices.items():

    plt.figure(figsize=(6,6))

    img.plot(cmap="RdYlGn")

    plt.title(name)

    plt.axis("off")

    plt.show()



import numpy as np
import pandas as pd

def extract_s2_features(dataset):

    blue  = dataset.B02.squeeze()
    green = dataset.B03.squeeze()
    red   = dataset.B04.squeeze()

    rededge = dataset.B05.squeeze()

    nir = dataset.B08.squeeze()

    swir1 = dataset.B11.squeeze()
    swir2 = dataset.B12.squeeze()

    features = {}

    features["NDVI"] = float(((nir-red)/(nir+red)).mean())

    features["EVI"] = float((
        2.5*((nir-red)/(nir+6*red-7.5*blue+1))
    ).mean())

    features["SAVI"] = float((
        1.5*((nir-red)/(nir+red+0.5))
    ).mean())

    features["MSAVI"] = float(((
        (2*nir+1) -
        np.sqrt((2*nir+1)**2-8*(nir-red))
    )/2).mean())

    features["GNDVI"] = float(((nir-green)/(nir+green)).mean())

    features["NDMI"] = float(((nir-swir1)/(nir+swir1)).mean())

    features["NDWI"] = float(((green-nir)/(green+nir)).mean())

    features["BSI"] = float(((
        (swir1+red)-(nir+blue)
    )/(
        (swir1+red)+(nir+blue)
    )).mean())

    features["NDRE"] = float(((nir-rededge)/(nir+rededge)).mean())

    features["CIre"] = float(((nir/rededge)-1).mean())

    features["Brightness"] = float(
        np.sqrt((red**2+nir**2)/2).mean()
    )

    features["ClayIndex"] = float(
        (swir1/swir2).mean()
    )

    return pd.DataFrame([features])



features = extract_s2_features(dataset)

features



In [6]:
import os
import time
import pandas as pd

# Configure GDAL HTTP timeouts to prevent indefinite hangs
os.environ["GDAL_HTTP_TIMEOUT"] = "15"
os.environ["GDAL_HTTP_MAX_RETRY"] = "3"
os.environ["GDAL_HTTP_RETRY_DELAY"] = "2"
os.environ["CPL_VSIL_CURL_ALLOWED_EXTENSIONS"] = ".tif,.tiff"

# Load labels
df_lucas = pd.read_csv("../data/features/lucas_labels.csv")

# DEBUG ONLY - 5 samples
df_lucas = df_lucas.head(5)

print("=" * 60)
print(f"Running DEBUG on {len(df_lucas)} points")
print(df_lucas["POINT_ID"].tolist())
print("=" * 60)

results = []
print("="*60)
print("Starting Sequential Execution")
print("="*60)

for _, row in df_lucas.iterrows():
    result = process_point(row)
    results.append(result)
    if result["status"] == "success":
        pid = result["data"]["POINT_ID"]
        print(f"Saving features for POINT_ID={pid}")

print("="*60)
print(f"Extracted {len(results)} results")
print("="*60)

# Save final debug output
df_final = pd.DataFrame([r["data"] for r in results if r["status"] == "success"])
os.makedirs("../data/features", exist_ok=True)
df_final.to_csv("../data/features/sentinel2_features.csv", index=False)
print("S2 Debug Merge Complete. Total rows:", len(df_final))


Running DEBUG on 5 points
[47862690, 47882704, 47982688, 48022702, 48062708]
Starting Sequential Execution
Starting POINT_ID=47862690


Searching STAC...


STAC search finished
Items found = 16
Loading raster...
Calling stac_load...
stac_load finished in 0.28 sec
Raster loaded
<xarray.Dataset> Size: 15kB
Dimensions:      (y: 23, x: 16, time: 1)
Coordinates:
  * y            (y) float64 184B 5.223e+06 5.223e+06 ... 5.222e+06 5.222e+06
  * x            (x) float64 128B 5.859e+05 5.859e+05 ... 5.861e+05 5.861e+05
  * time         (time) datetime64[ns] 8B 2018-07-07T09:50:29.024000
    spatial_ref  int32 4B 32633
Data variables:
    B02          (time, y, x) float32 1kB dask.array<chunksize=(1, 23, 16), meta=np.ndarray>
    B03          (time, y, x) float32 1kB dask.array<chunksize=(1, 23, 16), meta=np.ndarray>
    B04          (time, y, x) float32 1kB dask.array<chunksize=(1, 23, 16), meta=np.ndarray>
    B05          (time, y, x) float32 1kB dask.array<chunksize=(1, 23, 16), meta=np.ndarray>
    B06          (time, y, x) float32 1kB dask.array<chunksize=(1, 23, 16), meta=np.ndarray>
    B07          (time, y, x) float32 1kB dask.array<chunk

Aborting load due to failure while reading: https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/33/T/WN/2018/07/07/S2B_MSIL2A_20180707T095029_N0212_R079_T33TWN_20201011T130756.SAFE/GRANULE/L2A_T33TWN_A006969_20180707T095140/IMG_DATA/R20m/T33TWN_20180707T095029_B11_20m.tif?st=2026-07-17T09%3A55%3A11Z&se=2026-07-18T10%3A40%3A11Z&sp=rl&sv=2025-07-05&sr=c&skoid=9c8ff44a-6a2c-4dfb-b298-1c9212f64d9a&sktid=72f988bf-86f1-41af-91ab-2d7cd011db47&skt=2026-07-18T09%3A36%3A28Z&ske=2026-07-25T09%3A36%3A28Z&sks=b&skv=2025-07-05&sig=98nEqmLAj0oTYr/2aFTV9UnFcPcHz0PP0pSDmwpFZ6w%3D:1


Aborting load due to failure while reading: https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/33/T/WN/2018/07/07/S2B_MSIL2A_20180707T095029_N0212_R079_T33TWN_20201011T130756.SAFE/GRANULE/L2A_T33TWN_A006969_20180707T095140/IMG_DATA/R20m/T33TWN_20180707T095029_B05_20m.tif?st=2026-07-17T09%3A55%3A11Z&se=2026-07-18T10%3A40%3A11Z&sp=rl&sv=2025-07-05&sr=c&skoid=9c8ff44a-6a2c-4dfb-b298-1c9212f64d9a&sktid=72f988bf-86f1-41af-91ab-2d7cd011db47&skt=2026-07-18T09%3A36%3A28Z&ske=2026-07-25T09%3A36%3A28Z&sks=b&skv=2025-07-05&sig=98nEqmLAj0oTYr/2aFTV9UnFcPcHz0PP0pSDmwpFZ6w%3D:1


...

Computing NDVI...


Features extracted
Finished POINT_ID=48022702
Elapsed = 12.90 sec
Saving features for POINT_ID=48022702
Starting POINT_ID=48062708
Searching STAC...


STAC search finished
Items found = 48
Loading raster...
Calling stac_load...
stac_load finished in 0.03 sec
Raster loaded
<xarray.Dataset> Size: 15kB
Dimensions:      (y: 23, x: 16, time: 1)
Coordinates:
  * y            (y) float64 184B 5.239e+06 5.239e+06 ... 5.239e+06 5.239e+06
  * x            (x) float64 128B 6.07e+05 6.07e+05 ... 6.072e+05 6.072e+05
  * time         (time) datetime64[ns] 8B 2018-07-05T10:00:31.025000
    spatial_ref  int32 4B 32633
Data variables:
    B02          (time, y, x) float32 1kB dask.array<chunksize=(1, 23, 16), meta=np.ndarray>
    B03          (time, y, x) float32 1kB dask.array<chunksize=(1, 23, 16), meta=np.ndarray>
    B04          (time, y, x) float32 1kB dask.array<chunksize=(1, 23, 16), meta=np.ndarray>
    B05          (time, y, x) float32 1kB dask.array<chunksize=(1, 23, 16), meta=np.ndarray>
    B06          (time, y, x) float32 1kB dask.array<chunksize=(1, 23, 16), meta=np.ndarray>
    B07          (time, y, x) float32 1kB dask.array<chunksi

Computing NDVI...


Features extracted
Finished POINT_ID=48062708
Elapsed = 4.27 sec
Saving features for POINT_ID=48062708
Extracted 5 results
S2 Debug Merge Complete. Total rows: 5
